# 📈 Time Series with Pandas
### A Step-by-Step Lesson for Beginners

---
**Topics we will cover:**
1. What is a Time Series? + Loading data
2. Date formats & common pitfalls
3. DatetimeIndex & sorting
4. Date slicing
5. Date ranges & frequencies
6. Reindexing with a date range
7. Shifting
8. Resampling — Downsampling
9. Resampling — Upsampling + ffill / bfill
10. Interpolation
11. Rolling windows
12. Expanding windows
13. Aggregation with .agg()

## 🔧 Step 0 — Install required libraries
Run this cell once. You can skip it if these are already installed.

In [ ]:
!pip install pandas numpy matplotlib

## 📦 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
pd.options.display.float_format = '{:,.2f}'.format

print('Libraries loaded successfully!')

---
# 📌 Topic 1 — What is a Time Series? + Loading Data

A **time series** is data collected or recorded at regular time intervals.

Examples:
- Stock prices every day
- Temperature every hour
- Website visits every minute

In pandas, we use a **DatetimeIndex** — the row labels are dates instead of numbers.

**Key tips when loading:**
- `index_col='Date'` → makes the Date column the row label
- `parse_dates=True` → converts text dates into real datetime objects

In [ ]:
df = pd.read_csv('data/stocks.csv', index_col='Date', parse_dates=True)

print('Shape:', df.shape)
print('Index type:', type(df.index))
df.head()

In [ ]:
df.tail()

---
# 📌 Topic 2 — Date Formats & Common Pitfalls

⚠️ Very common beginner mistake!

These look similar but mean different things:
- `2020-03-01` → March 1st
- `2020-01-03` → January 3rd

Always check your CSV format before slicing.

In [ ]:
print('First date:', df.index[0])
print('Last date: ', df.index[-1])
print('Index dtype:', df.index.dtype)

In [ ]:
# Without parse_dates — dates stay as plain text strings
df_wrong = pd.read_csv('data/stocks.csv', index_col='Date')

print('Without parse_dates:', df_wrong.index.dtype)
print('With    parse_dates:', df.index.dtype)
# Text dates cannot be sliced, shifted, or resampled properly!

---
# 📌 Topic 3 — DatetimeIndex & Sorting

Before slicing, **always sort your index first**.

- Ascending → earliest date at the top
- Descending → latest date at the top

⚠️ In newer pandas, slicing a descending index with ascending dates raises a **KeyError** — not just an empty DataFrame!

The safest rule: **always sort ascending, then slice**. If you ever need to work with a descending index, use boolean masking with `.loc[]` instead.

In [ ]:
df = df.sort_index(ascending=True)

print('First row:', df.index[0])
print('Last row: ', df.index[-1])

In [ ]:
# Pitfall demo — newer pandas RAISES AN ERROR if you slice a descending index
# with ascending date order. The safest habit: always sort ascending first,
# then slice. That's what we do for the rest of this notebook.

df_desc = df.sort_index(ascending=False)

# Safe way to filter on a descending index → use boolean masking with .loc[]
jan_mask = (df_desc.index >= '2020-01-01') & (df_desc.index <= '2020-01-31')
jan_rows = df_desc.loc[jan_mask]

print('January rows found (descending index, boolean mask):', len(jan_rows))
print()
print('👉 Lesson: always sort ascending before using date slices.')
print('   Boolean masking works on any sort order and never raises errors.')

---
# 📌 Topic 4 — Date Slicing

With a sorted DatetimeIndex, slicing is very flexible:
- Specific dates, just a year or month, with a step, or using datetime variables

In [ ]:
# Slice between two specific dates (both ends inclusive)
df['2020-01-06':'2020-01-15']

In [ ]:
# Slice by year-month only — all rows in February 2020
# Use .loc[] so pandas knows we mean row labels, not column names
df.loc['2020-02']

In [ ]:
# Slice with a step — every 5th row in the range
df['2020-01':'2020-03':5]

In [ ]:
# Slice using Python datetime variables
from datetime import datetime

start = datetime(2020, 2, 1)
stop  = datetime(2020, 2, 14)

df[start:stop]

---
# 📌 Topic 5 — Date Ranges & Frequencies

`pd.date_range()` generates a sequence of dates at a chosen frequency.

**Two ways to call it:**
- Version 1: `start` + `end` + `freq`
- Version 2: `start` + `periods` + `freq`

**Common offset aliases:**

| Alias | Meaning |
|-------|---------|
| `D` | Calendar day |
| `B` | Business day |
| `W` | Weekly (Sunday) |
| `ME` | Month end |
| `BM` | Business month end |
| `QE` | Quarter end |
| `YE` | Year end |
| `h` | Hourly |
| `WOM-3FRI` | 3rd Friday of each month |

In [ ]:
# Version 1: start + end + freq
pd.date_range(start='2020-01-01', end='2020-01-31', freq='W')

In [ ]:
# Quarter-end dates
pd.date_range(start='2020-01-01', end='2020-12-31', freq='QE')

In [ ]:
# Anchored offset — Quarter ending in January
pd.date_range(start='2020-01-01', end='2020-12-31', freq='QE-JAN')

In [ ]:
# Anchored offset — 3rd Friday of every month
pd.date_range(start='2020-01-01', end='2020-06-30', freq='WOM-3FRI')

In [ ]:
# Sub-day frequency — every 4 hours
pd.date_range(start='2020-01-01', end='2020-01-02', freq='4h')

In [ ]:
# Version 2: start + periods + freq — 8 dates spaced 90 minutes apart
# pandas 2.x dropped compound strings like '1h30min'; use '90min' instead
pd.date_range(start='2020-01-01', periods=8, freq='90min')

---
# 📌 Topic 6 — Reindexing with a Date Range

Use a generated date range as labels to **reindex** an existing DataFrame.

- Date exists in DataFrame → shows data
- Date does NOT exist → shows `NaN`

Useful to align data to a specific calendar.

In [ ]:
# Generate business-month-end dates
# pandas 2.x renamed 'BM' → 'BME' (Business Month End)
biz_month_ends = pd.date_range(start='2020-01-01', end='2020-03-31', freq='BME')
print(biz_month_ends)

In [ ]:
# Reindex — matching dates show data, others show NaN
df.reindex(labels=biz_month_ends)

---
# 📌 Topic 7 — Shifting

**Shifting** slides data forward or backward along the time axis.

- `shift(1)`  → each row's value moves DOWN one row
- `shift(-1)` → each row's value moves UP one row
- The row that falls off the edge becomes `NaN`

**Practical use:** Compare today's price to yesterday's price.

In [ ]:
print('--- Original ---')
print(df['Close'].head())

In [ ]:
# Shift forward — first row becomes NaN
print('--- After shift(1) head ---')
print(df['Close'].shift(1).head())

In [ ]:
# Shift forward — last row becomes NaN (it fell off)
print('--- After shift(1) tail ---')
print(df['Close'].shift(1).tail())

In [ ]:
# Practical use: daily price change = today - yesterday
df['Daily Change'] = df['Close'] - df['Close'].shift(1)
df[['Close', 'Daily Change']].head(6)

---
# 📌 Topic 8 — Resampling: Downsampling

**Resampling** converts data from one frequency to another.

**Downsampling** = finer → coarser (Days → Months → Years)

Easier direction — you are **summarising** many rows into one. Must choose an aggregation function.

In [ ]:
# Daily data → monthly averages
df_monthly = df[['Open', 'Close']].resample(rule='ME').mean()
df_monthly

In [ ]:
# Daily data → weekly averages
df_weekly = df[['Open', 'Close']].resample(rule='W').mean()
df_weekly

In [ ]:
df['Close'].plot(label='Daily', alpha=0.5, figsize=(12, 4))
df_monthly['Close'].plot(label='Monthly avg', linewidth=2)
plt.title('Daily vs Monthly Average Close Price')
plt.legend()
plt.show()

---
# 📌 Topic 9 — Resampling: Upsampling + ffill / bfill

**Upsampling** = coarser → finer (Months → Days)

New rows that didn't exist start as `NaN`. Fill them with:
- **`ffill()`** — carry last known value forward
- **`bfill()`** — pull next known value backward

In [ ]:
print('Monthly data:')
print(df_monthly['Close'])

In [ ]:
# Upsample without filling — NaN everywhere except original rows
df_monthly['Close'].resample(rule='W').mean().head(10)

In [ ]:
# Forward fill — carry each month's value forward
print('Forward fill:')
print(df_monthly['Close'].resample(rule='W').ffill())

In [ ]:
# Backward fill — pull next month's value back
print('Backward fill:')
print(df_monthly['Close'].resample(rule='W').bfill())

---
# 📌 Topic 10 — Interpolation

Instead of copying a flat value (ffill/bfill), **interpolation** estimates missing values by drawing a curve through the known points.

- **Linear** — straight line
- **Quadratic** — slightly curved
- **Cubic** — smoothest curve

The chart below shows the difference visually.

In [ ]:
df_interp = pd.DataFrame()

# Upsample monthly → daily by reindexing (creates NaN gaps between known points)
daily_index = pd.date_range(start=df_monthly.index.min(), end=df_monthly.index.max(), freq='D')
close_daily = df_monthly['Close'].reindex(daily_index)

# Linear — fills gaps with a straight line between known points
df_interp['Linear']    = close_daily.interpolate(method='linear')

# Quadratic — fills gaps with a smooth curve (degree 2)
df_interp['Quadratic'] = close_daily.interpolate(method='quadratic')

# Cubic — fills gaps with a smoother curve (degree 3)
# Using numpy polyfit: works with any number of points, no scipy needed
import warnings
x_known = np.where(close_daily.notna())[0]
y_known = close_daily.dropna().values
x_all   = np.arange(len(close_daily))
with warnings.catch_warnings():
    warnings.simplefilter('ignore', np.exceptions.RankWarning)
    coeffs = np.polyfit(x_known, y_known, deg=3)
df_interp['Cubic'] = np.polyval(coeffs, x_all)

df_interp.head(10)

In [ ]:
df_interp.plot(figsize=(12, 4))
plt.title('Comparing Interpolation Methods (Monthly to Daily)')
plt.ylabel('Close Price')
plt.show()

---
# 📌 Topic 11 — Rolling Windows

A **rolling window** slides a fixed-size window across the data one row at a time.

- `rolling(window=7)` → looks at the last 7 rows at each step
- First `window - 1` rows are `NaN` (not enough data yet)

**Most common use:** Moving averages in stock analysis

In [ ]:
df['MA_7']  = df['Close'].rolling(window=7).mean()
df['MA_21'] = df['Close'].rolling(window=21).mean()

# Notice NaN at top until the window is full
df[['Close', 'MA_7', 'MA_21']].head(25)

In [ ]:
df['Close'].plot(label='Daily Close', alpha=0.4, figsize=(12, 4))
df['MA_7'].plot(label='7-day MA')
df['MA_21'].plot(label='21-day MA', linewidth=2)
plt.title('Close Price with 7-day and 21-day Moving Averages')
plt.legend()
plt.show()

---
# 📌 Topic 12 — Expanding Windows

An **expanding window** grows from the very first row to the current row.

- At row 1 → window has 1 value
- At row N → window has N values

Think of it as the **"running average since the beginning"**.

Useful for: cumulative averages, running totals, all-time highs.

In [ ]:
df['Expanding Mean'] = df['Close'].expanding().mean()

df[['Close', 'Expanding Mean']].head(10)

In [ ]:
df['Close'].plot(label='Daily Close', alpha=0.4, figsize=(12, 4))
df['MA_7'].plot(label='7-day Rolling Mean')
df['Expanding Mean'].plot(label='Expanding Mean', linewidth=2, linestyle='--')
plt.title('Rolling Window vs Expanding Window')
plt.legend()
plt.show()

---
# 📌 Topic 13 — Aggregation with .agg()

`.agg()` is a more flexible way to apply aggregation after resampling.

- Pass a string → `agg('mean')`
- Pass a list → `agg(['mean', 'max', 'min'])`
- Chain with column selection + date filter for powerful queries
- Use `.transpose()` to flip rows and columns

In [ ]:
# Option 1: direct .mean()
df['Close'].resample(rule='ME').mean()

In [ ]:
# Option 2: same result with .agg()
df['Close'].resample(rule='ME').agg('mean')

In [ ]:
# Store function name in a variable
func = 'mean'
df['Close'].resample(rule='ME').agg(func)

In [ ]:
# Multiple aggregation functions at once
funcs = ['mean', 'max', 'min']
df['Close'].resample(rule='ME').agg(funcs)

In [ ]:
# Most powerful form: columns + date filter + resample + multi-agg
funcs   = ['mean', 'max', 'min']
cols    = ['High', 'Low']
from_dt = '2020-01'
to_dt   = '2020-02'
freq    = 'W'

result = df[from_dt:to_dt][cols].resample(rule=freq).agg(funcs)
result

In [ ]:
# Transpose — sometimes easier to read
result.transpose()

---
# ✅ Summary

| # | Topic | Key function(s) |
|---|-------|-----------------|
| 1 | Loading time series data | `pd.read_csv(..., parse_dates=True)` |
| 2 | Date format pitfalls | Always check `dtype` of index |
| 3 | DatetimeIndex & sorting | `.sort_index()` |
| 4 | Date slicing | `df['2020-01':'2020-03']` |
| 5 | Date ranges & frequencies | `pd.date_range()` + offset aliases |
| 6 | Reindexing | `.reindex(labels=date_range)` |
| 7 | Shifting | `.shift(n)` |
| 8 | Downsampling | `.resample().mean()` |
| 9 | Upsampling + fill | `.resample().ffill()` / `.bfill()` |
| 10 | Interpolation | `.resample().interpolate(method=...)` |
| 11 | Rolling windows | `.rolling(window=n).mean()` |
| 12 | Expanding windows | `.expanding().mean()` |
| 13 | Aggregation | `.resample().agg([...])` |

---
🎉 **End of Lesson**